# 05 — Modeling: Dự đoán nguy cơ học tập kém (`Risk_Level`)

**Mục tiêu:** xây dựng, so sánh và đánh giá mô hình phân loại 3 lớp **Low / Medium / High** từ các yếu tố học tập, sinh hoạt, gia đình và nhà trường của sinh viên.

**Đầu vào:** `data/processed/dataset_with_risk.csv` (kết quả của `03_data_cleaning.ipynb`)

**Đầu ra:**
- `models/`: `best_model.pkl`, `selected_features.pkl`, `best_model_info.json`, `model_comparison.csv`, `tuning_results.json`
- `reports/`: `confusion_matrix.csv/.png`, `evaluation_report.json`, `feature_importance.csv`, `predictions.csv`

**Nguyên tắc:**
1. `Exam_Score` tạo ra nhãn nên **không được làm feature** (tránh data leakage).
2. Chọn mô hình và tinh chỉnh **chỉ dựa trên cross-validation của tập train**. Tập test chỉ dùng một lần để báo cáo kết quả cuối.
3. Ưu tiên **F1-macro** và **recall lớp High** (không bỏ sót sinh viên nguy cơ cao), không chỉ accuracy.

> Chạy từ trên xuống dưới. Nếu cần chạy lại, chọn **Runtime → Restart session and run all**.

## 1. Cấu hình và đọc dữ liệu

In [ ]:
import glob
import json
import time
import warnings
from datetime import datetime
from pathlib import Path

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import sklearn
from sklearn.dummy import DummyClassifier
from sklearn.ensemble import GradientBoostingClassifier, RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (accuracy_score, classification_report, confusion_matrix,
                             f1_score, make_scorer, recall_score)
from sklearn.model_selection import (GridSearchCV, StratifiedKFold, cross_validate,
                                     train_test_split)
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.tree import DecisionTreeClassifier

warnings.filterwarnings("ignore")

SEED = 42
TEST_SIZE = 0.2
TARGET = "Risk_Level"
LABELS = ["Low", "Medium", "High"]       # thứ tự hiển thị cố định
DROP = ["Exam_Score", TARGET]            # Exam_Score tạo ra nhãn, không được làm feature


def find_data():
    """Tìm dataset_with_risk.csv trong repo hoặc Google Drive."""
    for p in [Path("data/processed/dataset_with_risk.csv"),
              Path("../data/processed/dataset_with_risk.csv"),
              Path("dataset_with_risk.csv")]:
        if p.exists():
            return p
    hits = glob.glob("/content/drive/MyDrive/**/dataset_with_risk*.csv", recursive=True)
    if hits:
        return Path(hits[0])
    raise FileNotFoundError("Không tìm thấy dataset_with_risk.csv. Hãy chạy 03_data_cleaning "
                            "trước hoặc tải file lên cùng thư mục với notebook.")


DATA_PATH = find_data()
ROOT = DATA_PATH.parent.parent.parent if DATA_PATH.parent.name == "processed" else Path(".")
MODELS_DIR, REPORTS_DIR = ROOT / "models", ROOT / "reports"
MODELS_DIR.mkdir(parents=True, exist_ok=True)
REPORTS_DIR.mkdir(parents=True, exist_ok=True)
print("Dữ liệu:", DATA_PATH)
print("Lưu mô hình vào:", MODELS_DIR.resolve())
print("Lưu báo cáo vào:", REPORTS_DIR.resolve())

## 2. Kiểm tra chất lượng dữ liệu đầu vào (Quality Gate)

Dừng ngay nếu dữ liệu có vấn đề, thay vì để lỗi lan sang mô hình.

In [ ]:
df = pd.read_csv(DATA_PATH)

assert df.isna().sum().sum() == 0, "Dữ liệu còn ô thiếu: chạy lại 03_data_cleaning"
assert df["Exam_Score"].max() <= 100, "Còn điểm thi vượt thang 100"
assert set(df[TARGET].unique()) == set(LABELS), "Nhãn Risk_Level không đúng"
assert df.duplicated().sum() == 0, "Có dòng trùng lặp"

print("Kích thước:", df.shape)
print("\nPhân bố nhãn:")
print(df[TARGET].value_counts().reindex(LABELS))
print((df[TARGET].value_counts(normalize=True).reindex(LABELS) * 100).round(2).astype(str) + " %")

## 3. Chuẩn bị feature và chia dữ liệu

- Loại `Exam_Score` và `Risk_Level` khỏi feature.
- Chia train/test **phân tầng** (`stratify=y`) để tỷ lệ 3 lớp giống nhau ở hai tập.
- Một ô chia dữ liệu duy nhất: mọi bước sau dùng lại `X_train`, `X_test`.

In [ ]:
X = df.drop(columns=DROP)
X = X.astype({c: int for c in X.select_dtypes("bool").columns}).astype(float)
y = df[TARGET].astype(str)
FEATURES = list(X.columns)

assert not set(DROP) & set(FEATURES), "Feature chứa cột gây leakage"

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=TEST_SIZE, random_state=SEED, stratify=y)

print(f"Số feature: {len(FEATURES)}")
print(f"Train: {X_train.shape} | Test: {X_test.shape}")
print("\nTỷ lệ nhãn (train / test):")
print(pd.DataFrame({"train": y_train.value_counts(normalize=True),
                    "test": y_test.value_counts(normalize=True)}).reindex(LABELS).round(4))

## 4. Mô hình cơ sở (baseline)

Mô hình luôn đoán lớp phổ biến nhất. Mọi mô hình thật phải vượt xa mức này thì mới có ý nghĩa.

In [ ]:
dummy = DummyClassifier(strategy="most_frequent").fit(X_train, y_train)
pred_dummy = dummy.predict(X_test)
baseline = {"accuracy": accuracy_score(y_test, pred_dummy),
            "f1_macro": f1_score(y_test, pred_dummy, average="macro")}
print(f"Baseline (luôn đoán '{dummy.classes_[0]}'): "
      f"accuracy = {baseline['accuracy']:.4f} | F1-macro = {baseline['f1_macro']:.4f}")

## 5. So sánh các mô hình bằng cross-validation (chỉ trên tập train)

Các mô hình đều nằm trong `Pipeline(StandardScaler → mô hình)` để scaler chỉ được fit trên phần train của từng fold (không rò rỉ).

In [ ]:
def pipe(est):
    return Pipeline([("scaler", StandardScaler()), ("model", est)])


def recall_high(y_true, y_pred):
    return recall_score(y_true, y_pred, labels=["High"], average="macro", zero_division=0)


candidates = {
    "Logistic Regression": pipe(LogisticRegression(max_iter=1000)),
    "Decision Tree": pipe(DecisionTreeClassifier(random_state=SEED)),
    "KNN": pipe(KNeighborsClassifier()),
    "Random Forest": pipe(RandomForestClassifier(n_estimators=200, random_state=SEED, n_jobs=-1)),
    "Gradient Boosting": pipe(GradientBoostingClassifier(random_state=SEED)),
}
cv = StratifiedKFold(5, shuffle=True, random_state=SEED)
scoring = {"accuracy": "accuracy", "f1_macro": "f1_macro",
           "recall_high": make_scorer(recall_high)}

rows = []
for name, model in candidates.items():
    t0 = time.time()
    cvr = cross_validate(model, X_train, y_train, cv=cv, scoring=scoring)
    pred = model.fit(X_train, y_train).predict(X_test)
    rows.append({
        "model": name,
        "cv_accuracy": cvr["test_accuracy"].mean(),
        "cv_f1_macro": cvr["test_f1_macro"].mean(),
        "cv_f1_macro_std": cvr["test_f1_macro"].std(),
        "cv_recall_high": cvr["test_recall_high"].mean(),
        "test_accuracy": accuracy_score(y_test, pred),
        "test_f1_macro": f1_score(y_test, pred, average="macro"),
        "test_recall_high": recall_high(y_test, pred),
        "seconds": round(time.time() - t0, 1),
    })
    print(f"{name:20s} CV F1-macro = {rows[-1]['cv_f1_macro']:.4f}")

comparison = (pd.DataFrame(rows).sort_values("cv_f1_macro", ascending=False)
              .reset_index(drop=True))
print()
print(comparison.round(4).to_string(index=False))

In [ ]:
fig, ax = plt.subplots(figsize=(8, 3.8))
ax.barh(comparison["model"][::-1], comparison["cv_f1_macro"][::-1],
        xerr=comparison["cv_f1_macro_std"][::-1], color="#4C78A8")
ax.axvline(baseline["f1_macro"], color="gray", linestyle="--", label="Baseline")
ax.set_xlabel("CV F1-macro (tập train, 5-fold)")
ax.set_title("So sánh các mô hình")
ax.legend()
fig.tight_layout()
plt.show()

## 6. Tinh chỉnh siêu tham số (Grid Search)

Chỉ tinh chỉnh 2 mô hình tốt nhất theo CV, dùng F1-macro. Vẫn chỉ dùng tập train.

In [ ]:
grids = {
    "Logistic Regression": {"model__C": [0.01, 0.1, 1, 10, 100]},
    "Decision Tree": {"model__max_depth": [3, 5, 8, 12], "model__min_samples_leaf": [1, 5, 20]},
    "KNN": {"model__n_neighbors": [5, 11, 21, 31]},
    "Random Forest": {"model__max_depth": [None, 10], "model__min_samples_leaf": [1, 3]},
    "Gradient Boosting": {"model__n_estimators": [100, 200], "model__max_depth": [2, 3]},
}
TUNE_TOP = 2

tuning, tuned = {}, {}
for name in comparison["model"].head(TUNE_TOP):
    gs = GridSearchCV(candidates[name], grids[name], cv=cv, scoring="f1_macro", n_jobs=1)
    gs.fit(X_train, y_train)
    base = float(comparison.loc[comparison["model"] == name, "cv_f1_macro"].iloc[0])
    tuning[name] = {
        "param_grid": {k: list(v) for k, v in grids[name].items()},
        "best_params": {k: (v if isinstance(v, (int, float, str, type(None))) else str(v))
                        for k, v in gs.best_params_.items()},
        "baseline_cv_f1_macro": round(base, 4),
        "tuned_cv_f1_macro": round(float(gs.best_score_), 4),
        "n_candidates": int(len(gs.cv_results_["params"])),
    }
    tuned[name] = gs.best_estimator_
    print(f"{name}: CV F1 {base:.4f} -> {gs.best_score_:.4f} | {gs.best_params_}")

## 7. Chọn mô hình cuối và đánh giá trên tập test

Mô hình được chọn theo **CV F1-macro cao nhất sau tuning** (không nhìn tập test). Tập test dùng đúng một lần ở bước này.

In [ ]:
best_name = max(tuning, key=lambda n: tuning[n]["tuned_cv_f1_macro"])
best = tuned[best_name]                      # GridSearchCV đã refit trên toàn bộ tập train
y_pred = pd.Series(best.predict(X_test), index=X_test.index)

report = classification_report(y_test, y_pred, labels=LABELS, output_dict=True, zero_division=0)
metrics = {
    "accuracy": float(accuracy_score(y_test, y_pred)),
    "f1_macro": float(f1_score(y_test, y_pred, labels=LABELS, average="macro")),
    "recall_high": float(report["High"]["recall"]),
}
print(f"Mô hình được chọn: {best_name} | {tuning[best_name]['best_params']}")
print(f"Accuracy = {metrics['accuracy']:.4f} | F1-macro = {metrics['f1_macro']:.4f} "
      f"| Recall High = {metrics['recall_high']:.4f}\n")
print(classification_report(y_test, y_pred, labels=LABELS, digits=3, zero_division=0))

### Confusion matrix

In [ ]:
cm = confusion_matrix(y_test, y_pred, labels=LABELS)
cm_df = pd.DataFrame(cm, index=LABELS, columns=LABELS)
cm_df.index.name = "actual\\predicted"
cm_norm = cm / cm.sum(axis=1, keepdims=True)

fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))
for ax, data, title, fmt in ((axes[0], cm, "Số lượng", "d"),
                             (axes[1], cm_norm, "Chuẩn hóa theo hàng (recall)", ".2f")):
    im = ax.imshow(data, cmap="Blues")
    ax.set_xticks(range(3), LABELS)
    ax.set_yticks(range(3), LABELS)
    ax.set_xlabel("Dự đoán")
    ax.set_ylabel("Thực tế")
    ax.set_title(title)
    for i in range(3):
        for j in range(3):
            ax.text(j, i, format(data[i, j], fmt), ha="center", va="center",
                    color="white" if data[i, j] > data.max() / 2 else "black")
    fig.colorbar(im, ax=ax, fraction=0.046)
fig.suptitle("Confusion Matrix - Risk_Level")
fig.tight_layout()
fig.savefig(REPORTS_DIR / "confusion_matrix.png", dpi=150)
plt.show()

print(cm_df)
print(f"\nSinh viên High bị đoán nhầm thành Low   : {cm_df.loc['High', 'Low']}")
print(f"Sinh viên High bị đoán nhầm thành Medium: {cm_df.loc['High', 'Medium']}")

### Phân tích lỗi

Nhãn được cắt từ `Exam_Score` tại hai ngưỡng (Q1 và Q3), nên các sinh viên điểm sát ngưỡng rất dễ bị xếp sai lớp. Kiểm tra xem lỗi có tập trung ở vùng này không.

In [ ]:
q1, q3 = df["Exam_Score"].quantile([0.25, 0.75])
wrong_idx = y_test.index[y_pred.loc[y_test.index].values != y_test.values]
wrong_scores = df.loc[wrong_idx, "Exam_Score"]
near = ((wrong_scores - q1).abs() <= 2) | ((wrong_scores - q3).abs() <= 2)

print(f"Ngưỡng điểm: Q1 = {q1:.0f}, Q3 = {q3:.0f}")
print(f"Số dự đoán sai: {len(wrong_idx)} / {len(y_test)} ({len(wrong_idx) / len(y_test):.1%})")
print(f"Trong số sai, tỷ lệ có điểm cách ngưỡng không quá 2 điểm: {near.mean():.1%}")
print("\nPhân bố Exam_Score của các ca dự đoán sai:")
print(wrong_scores.describe().round(2))

## 8. Yếu tố ảnh hưởng (Feature Importance)

Với Logistic Regression trên dữ liệu đã chuẩn hóa, độ lớn hệ số cho biết mức ảnh hưởng, dấu của hệ số cho biết chiều ảnh hưởng theo từng lớp.

In [ ]:
final_est = best.named_steps["model"]
fi = pd.DataFrame({"feature": FEATURES})
if hasattr(final_est, "coef_"):
    coef = np.atleast_2d(final_est.coef_)
    fi["importance"] = np.abs(coef).mean(axis=0)
    for k, c in enumerate(final_est.classes_):
        fi[f"coef_{c}"] = coef[k]
    method = "mean_abs_coef (hệ số trên dữ liệu đã chuẩn hóa)"
elif hasattr(final_est, "feature_importances_"):
    fi["importance"] = final_est.feature_importances_
    method = "feature_importances_"
else:
    from sklearn.inspection import permutation_importance
    r = permutation_importance(best, X_test, y_test, n_repeats=10, random_state=SEED)
    fi["importance"] = r.importances_mean
    method = "permutation"
fi["method"] = method
fi = fi.sort_values("importance", ascending=False).reset_index(drop=True)

top = fi.head(10)[::-1]
fig, ax = plt.subplots(figsize=(8, 4.5))
ax.barh(top["feature"], top["importance"], color="#59A14F")
ax.set_xlabel("Mức quan trọng (" + method.split(" ")[0] + ")")
ax.set_title("Top 10 yếu tố ảnh hưởng")
fig.tight_layout()
plt.show()

print(fi.drop(columns="method").round(3).head(10).to_string(index=False))

## 9. Lưu mô hình và báo cáo

Lưu **sau khi đã fit**, kèm danh sách feature đúng thứ tự cột để dashboard và script dùng lại.

In [ ]:
# --- models/ ---
joblib.dump(best, MODELS_DIR / "best_model.pkl")
joblib.dump(FEATURES, MODELS_DIR / "selected_features.pkl")
comparison.round(4).to_csv(MODELS_DIR / "model_comparison.csv", index=False)
with open(MODELS_DIR / "tuning_results.json", "w", encoding="utf-8") as f:
    json.dump({"scoring": "f1_macro", "cv_folds": 5, "random_state": SEED, "results": tuning},
              f, ensure_ascii=False, indent=2)

info = {
    "best_model": best_name,
    "pipeline": " + ".join(type(s).__name__ for _, s in best.steps),
    "best_params": tuning[best_name]["best_params"],
    "selection_rule": "CV F1-macro cao nhất trên tập train (5-fold, phân tầng); tập test chỉ dùng để báo cáo",
    "cv_f1_macro": tuning[best_name]["tuned_cv_f1_macro"],
    "test_metrics": {k: round(v, 4) for k, v in metrics.items()},
    "baseline": {k: round(v, 4) for k, v in baseline.items()},
    "n_train": int(len(X_train)), "n_test": int(len(X_test)),
    "test_size": TEST_SIZE, "random_state": SEED, "stratify": True,
    "n_features": len(FEATURES), "features": FEATURES, "excluded_columns": DROP,
    "sklearn_version": sklearn.__version__,
    "trained_at": datetime.now().strftime("%Y-%m-%d %H:%M"),
}
with open(MODELS_DIR / "best_model_info.json", "w", encoding="utf-8") as f:
    json.dump(info, f, ensure_ascii=False, indent=2)

# --- reports/ ---
cm_df.to_csv(REPORTS_DIR / "confusion_matrix.csv")
eval_report = {
    "model": info["pipeline"], "n_test": int(len(y_test)), "test_size": TEST_SIZE,
    "random_state": SEED, "features": FEATURES, **metrics,
    "per_class": {k: report[k] for k in LABELS},
    "macro_avg": report["macro avg"], "weighted_avg": report["weighted avg"],
    "high_predicted_as_low": int(cm_df.loc["High", "Low"]),
    "high_predicted_as_medium": int(cm_df.loc["High", "Medium"]),
}
with open(REPORTS_DIR / "evaluation_report.json", "w", encoding="utf-8") as f:
    json.dump(eval_report, f, ensure_ascii=False, indent=2)
fi.to_csv(REPORTS_DIR / "feature_importance.csv", index=False)

pred_df = pd.DataFrame({"actual": y_test.values, "predicted": y_pred.values}, index=X_test.index)
pred_df["correct"] = pred_df["actual"] == pred_df["predicted"]
proba = best.predict_proba(X_test)
for k, c in enumerate(best.classes_):
    pred_df[f"proba_{c}"] = proba[:, k]
pred_df.index.name = "row_index"
pred_df.to_csv(REPORTS_DIR / "predictions.csv")

print("Đã lưu:", sorted(p.name for p in MODELS_DIR.iterdir() if p.is_file()))
print("Đã lưu:", sorted(p.name for p in REPORTS_DIR.iterdir() if p.is_file()))

### Kiểm tra lại sau khi lưu

In [ ]:
from sklearn.utils.validation import check_is_fitted

loaded = joblib.load(MODELS_DIR / "best_model.pkl")
feats = joblib.load(MODELS_DIR / "selected_features.pkl")

check_is_fitted(loaded)                                   # lỗi nếu lưu trước khi fit
assert "Exam_Score" not in feats and TARGET not in feats  # không leakage
assert list(loaded.feature_names_in_) == list(feats)
assert (loaded.predict(X_test[feats]) == y_pred.values).all()
print("OK: mô hình đã fit, không leakage, dự đoán khớp với kết quả đánh giá.")

## 10. Tóm tắt kết quả

In [ ]:
top3 = ", ".join(fi["feature"].head(3))
print(f"Mô hình được chọn : {best_name} {tuning[best_name]['best_params']}")
print(f"So với baseline   : accuracy {baseline['accuracy']:.1%} -> {metrics['accuracy']:.1%}")
print(f"F1-macro (test)   : {metrics['f1_macro']:.3f}")
print(f"Recall lớp High   : {metrics['recall_high']:.1%} "
      f"({cm_df.loc['High', 'High']}/{cm_df.loc['High'].sum()} sinh viên High được phát hiện)")
print(f"High bị đoán thành Low: {cm_df.loc['High', 'Low']}")
print(f"3 yếu tố ảnh hưởng nhất: {top3}")

## 11. Nhận xét và giới hạn

**Nhận xét**
- Mô hình tuyến tính (Logistic Regression) **vượt xa** các mô hình cây và KNN. Điều này phù hợp với dữ liệu: nhãn sinh từ `Exam_Score`, mà điểm thi phụ thuộc gần như tuyến tính vào các biến đầu vào. Vì vậy mô hình đơn giản, dễ giải thích là đủ tốt cho bài toán cảnh báo sớm; các mô hình phức tạp hơn không mang lại lợi ích.
- Với cảnh báo sớm, lỗi đáng lo nhất là **bỏ sót** sinh viên High (đoán thành Low). Xem số liệu ở mục 7 và 10.
- Lỗi chủ yếu xảy ra ở các sinh viên **sát ngưỡng điểm** giữa hai lớp (mục phân tích lỗi), đây là hạn chế tự nhiên của cách đặt nhãn theo quartile, không phải do mô hình thiếu khả năng.

**Giới hạn**
1. Độ chính xác cao phản ánh cấu trúc của bộ dữ liệu (nhiều khả năng là dữ liệu tổng hợp, cần xác nhận), **không** đảm bảo hiệu quả tương tự trên sinh viên thật.
2. Nhãn mang tính **tương đối** (quartile), "High" nghĩa là thuộc nhóm điểm thấp nhất trong mẫu, không phải rớt môn.
3. Dữ liệu cắt ngang, không có yếu tố thời gian; chưa có các biến như `Stress`, `Social Media`, `Part-time Job`.
4. Mô hình dùng cả các biến nhạy cảm (`Gender`, `Family_Income`, `Learning_Disabilities`). Khi áp dụng thực tế cần xem xét tính công bằng và chỉ dùng kết quả để **hỗ trợ** sinh viên, không để đánh giá hay phân biệt đối xử.

**Bước tiếp theo:** dùng `models/best_model.pkl` và `selected_features.pkl` cho dashboard (`app.py`); viết `docs/model_report.md` từ các file trong `reports/`.